In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg ,udf
from pyspark.sql.types import StringType, IntegerType
import time
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (1, "Alice", "Engineering", 75000, 25),
    (2, "Bob", "Marketing", 60000, 30),
    (3, "Charlie", "Engineering", 80000, 35),
    (4, "David", "Sales", 65000, 28),
    (5, "Eve", "Marketing", 70000, 32)
]
columns = ["Id", "Name", "Department", "Salary", "Age"]
df = spark.createDataFrame(data, columns);

In [4]:
df_select=df.select("Name","Department","Salary")

In [5]:
df_select.show()

+-------+-----------+------+
|   Name| Department|Salary|
+-------+-----------+------+
|  Alice|Engineering| 75000|
|    Bob|  Marketing| 60000|
|Charlie|Engineering| 80000|
|  David|      Sales| 65000|
|    Eve|  Marketing| 70000|
+-------+-----------+------+



In [6]:
df_filter=df.filter(col("Salary")>65000)


In [7]:
df_filter.show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  1|  Alice|Engineering| 75000| 25|
|  3|Charlie|Engineering| 80000| 35|
|  5|    Eve|  Marketing| 70000| 32|
+---+-------+-----------+------+---+



In [8]:
df_with_col=df.withColumn("Bonus",col("Salary")*0.10)

In [8]:
df_with_col.show()

NameError: name 'df_with_col' is not defined

In [10]:
df.write.mode("overwrite").csv("output/employeeTable",header=True)

In [9]:
df.toPandas().to_csv(
    "output/employeeTableNew.csv",
    index=False
)

26/06/13 06:05:23 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [10]:
df=spark.read \
    .option("header",True) \
    .csv("output/employeeTable")

In [11]:
df.createOrReplaceTempView("employees")

In [12]:
result=spark.sql("""SELECT * FROM employees WHERE Salary>70000""").show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  3|Charlie|Engineering| 80000| 35|
|  1|  Alice|Engineering| 75000| 25|
+---+-------+-----------+------+---+



In [13]:
df=spark.range(0,1000000).withColumn("value",col("id")%1000)

In [14]:
print("before Partition",df.rdd.getNumPartitions())

before Partition 2


In [15]:
df_repartion=df.repartition(10)

In [16]:
print(df_repartion.rdd.getNumPartitions())

[Stage 8:>                                                          (0 + 2) / 2]

10


In [17]:
df_colesced=df_repartion.coalesce(2)

In [18]:
print(df_colesced.rdd.getNumPartitions())

[Stage 9:>                                                          (0 + 2) / 2]

2


In [19]:
optimized_df = df.filter(col("value") > 500).filter(col("id") < 5000000).select("id", "value")

In [20]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#105L, (id#105L % 1000) AS value#107L]
+- *(1) Filter (((id#105L % 1000) > 500) AND (id#105L < 5000000))
   +- *(1) Range (0, 1000000, step=1, splits=2)




In [25]:
start_time = time.time()

count_uncached = optimized_df.count() #Action triggers DAG

end_time = time.time()

print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds")

1. Optimized Execution | Count: 499000 | Time: 0.3502 seconds
